# Codabench Competition Tagger — Starting Kit

This notebook demonstrates the end-to-end workflow:
1. Load the sample data
2. Binarize the multi-label targets
3. Instantiate the baseline model and run a quick evaluation
4. Format predictions for submission

**Submission**: zip `model.py` at the archive root and upload to the competition.

In [ ]:
import os, sys, csv
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold

print('scikit-learn and numpy loaded OK')

In [ ]:
# ---------------------------------------------------------------------------
# Load the 10-row sample from the starting kit
#
# The data is split the same way as in the competition:
#   sample_input_data.csv     -- column: text
#   sample_reference_data.csv -- columns: ML_sector, Field_sector
# Both are ';'-separated and aligned row by row. Inside a label cell,
# several labels are separated by ','.
# ---------------------------------------------------------------------------
# Jupyter runs a notebook from its own directory, so the sample files and
# model.py sit right here at the root of the starting kit.
HERE = os.getcwd()
SAMPLE_INPUT_PATH = os.path.join(HERE, 'sample_input_data.csv')
SAMPLE_REF_PATH = os.path.join(HERE, 'sample_reference_data.csv')

with open(SAMPLE_INPUT_PATH, 'r', encoding='utf-8', newline='') as f:
    texts = [row['text'] for row in csv.DictReader(f, delimiter=';')]

ml_raw, field_raw = [], []
with open(SAMPLE_REF_PATH, 'r', encoding='utf-8', newline='') as f:
    for row in csv.DictReader(f, delimiter=';'):
        ml_raw.append([x.strip() for x in row['ML_sector'].split(',') if x.strip()])
        field_raw.append([x.strip() for x in row['Field_sector'].split(',') if x.strip()])

assert len(texts) == len(ml_raw) == len(field_raw), 'input and reference rows must align'

print(f'Loaded {len(texts)} samples')
print('Example text:', texts[0][:80], '...')
print('Example ML_sector labels:', ml_raw[0])
print('Example Field_sector labels:', field_raw[0])

In [ ]:
# ---------------------------------------------------------------------------
# Binarize labels using the fixed taxonomy (same as scoring program)
# ---------------------------------------------------------------------------
# ML_LABELS = [
#     'Natural Language Processing', 'Computer Vision',
#     'Tabular / Structured Data', 'Time Series & Forecasting',
#     'Reinforcement Learning', 'Generative Models', 'Graph Learning',
#     'Federated & Privacy-Preserving Learning',
#     'AutoML & Neural Architecture Search', 'Multimodal Learning',
# ]
# FIELD_LABELS = [
#     'Healthcare & Medicine', 'Biology & Bioinformatics',
#     'Climate & Environment', 'Finance & Economics',
#     'Agriculture & Food Science', 'Autonomous Systems & Robotics',
#     'Social Sciences & Humanities', 'Cybersecurity',
#     'Education & Learning Sciences', 'Materials & Physical Sciences',
# ]

ML_labels = [
    "NLP / Text", "Computer Vision", "Speech / Audio",
    "Time Series / Forecasting", "Graph / Networks",
    "Reinforcement Learning", "Generative / LLM",
    "Tabular / Classical ML", "ML / AutoML / HPO / NAS",
    "Algorithmics / non-ML", "Other"
]
FIELD_LABELS = [
    "Healthcare / Biology", "Climate / Energy",
    "E-commerce / Retail / Finance", "Security & Privacy",
    "Robotics & Autonomous", "Agriculture & Food",
    "Transportation & Mobility", "Hard Sciences / Mathematics",
    "Human Sciences", "Media / Social", "Other"
]

mlb_ml    = MultiLabelBinarizer(classes=ML_labels)
mlb_field = MultiLabelBinarizer(classes=FIELD_LABELS)
y_ml    = mlb_ml.fit_transform(ml_raw)
y_field = mlb_field.fit_transform(field_raw)

print(f'y_ml shape: {y_ml.shape}   (n_samples x 11 ML labels)')
print(f'y_field shape: {y_field.shape}   (n_samples x 11 Field labels)')

In [ ]:
# ---------------------------------------------------------------------------
# Load and run the baseline Model
# ---------------------------------------------------------------------------
# model.py sits next to this notebook in the starting kit.
sys.path.insert(0, HERE)
from model import Model

# Quick 2-fold demo on the 10-row sample
kf = KFold(n_splits=2, shuffle=True, random_state=42)
X = np.array(texts, dtype=object)
fold_scores = []

for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X)):
    m = Model()
    m.fit(X[train_idx].tolist(), y_ml[train_idx], y_field[train_idx])
    y_ml_pred, y_field_pred = m.predict(X[test_idx].tolist())

    f1_ml    = f1_score(y_ml[test_idx],    y_ml_pred,    average='macro', zero_division=0)
    f1_field = f1_score(y_field[test_idx], y_field_pred, average='macro', zero_division=0)
    score = (f1_ml + f1_field) / 2.0
    fold_scores.append(score)
    print(f'Fold {fold_idx+1}: f1_ml={f1_ml:.3f}  f1_field={f1_field:.3f}  score={score:.3f}')

print(f'\nMean score (demo, 2-fold on 10 rows): {np.mean(fold_scores):.3f}')
print('Note: the competition uses 4-fold CV on 200 rows — expect different numbers there.')

## Next steps

1. Copy `model_template.py` to `model.py` and implement `fit` and `predict`.
2. Test locally against `sample_input_data.csv` and `sample_reference_data.csv`.
3. Create your submission: `zip submission.zip model.py`.
4. Upload `submission.zip` on the competition page.

The scoring program will train and evaluate your model on the full 200-sample dataset
using 4-fold cross-validation and report Mean Macro F1 on the leaderboard.